# 🚀 Nitro Google Maps Scraper (Colab Edition)
Welcome! This notebook allows you to run high-performance Google Maps scraping directly in Google Colab using the Nitro (Rod) engine.

**Note:** This uses a lightweight, parallel Go engine optimized for cloud environments.

## ⚙️ 1. Setup Environment
Run this cell to install required Linux libraries and download the Nitro Engine.

In [ ]:
# Install Python dependencies
!pip install pandas requests

# Install System dependencies (Chromium and friends)
!apt-get update
!apt-get install -y chromium-browser libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxext6 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2 libxshmfence1 libglu1-mesa ca-certificates fonts-liberation libappindicator3-1 libfontconfig1 libsecret-1-0 libxss1 lsb-release

# Download Nitro Engine (Rod Version)
import os
import urllib.request
import platform

engine_url = "https://github.com/gosom/google-maps-scraper/releases/download/v1.10.1/google_maps_scraper-1.10.1-rod-linux-amd64"
binary_path = "./gmaps_scraper"

print("📥 Downloading Nitro Engine...")
urllib.request.urlretrieve(engine_url, binary_path)
os.chmod(binary_path, 0o755)
print("✅ Setup Complete!")

## 💎 2. Define Scraper Function

In [ ]:
import subprocess
import json
import uuid
import pandas as pd

def scrape_gmaps(queries, depth=10, concurrency=1):
    run_id = str(uuid.uuid4())[:8]
    input_file = f"queries_{run_id}.txt"
    output_file = f"results_{run_id}.json"
    
    with open(input_file, 'w') as f:
        f.write("\n".join(queries))
    
    print(f"🚀 Starting extraction for {len(queries)} terms...")
    cmd = [
        "./gmaps_scraper",
        "-input", input_file,
        "-results", output_file,
        "-json",
        "-depth", str(depth),
        "-c", str(concurrency),
        "-exit-on-inactivity", "2m"
    ]
    
    try:
        subprocess.run(cmd, check=True)
        if os.path.exists(output_file):
            with open(output_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            df = pd.DataFrame(data)
            print(f"✅ Successfully extracted {len(df)} leads!")
            return df
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
    return pd.DataFrame()

## 🚀 3. Run Extraction
Add your search terms here!

In [ ]:
my_queries = [
    "Restaurant in Gujranwala",
    "Dentist in London"
]

results_df = scrape_gmaps(my_queries, depth=10, concurrency=1)
results_df.head()

## 📂 4. Download Results

In [ ]:
if not results_df.empty:
    results_df.to_csv("leads_export.csv", index=False)
    from google.colab import files
    files.download("leads_export.csv")
else:
    print("No results to download.")